# Stage 5 — Fine-tuning BERT as a regressor

**Goal.** Explore the BERT regression setup (target scaling, long-text strategy, early stopping on validation MAE) and its training curves.

Logic: `src/train_bert.py`. Training runs as a script (`python -m src.train_bert`), not in this notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

## Setup

`src/train_bert.py` fine-tunes **distilbert-base-uncased** as a regressor:

- **Model and data.** `AutoModelForSequenceClassification(num_labels=1, problem_type="regression")`, trained on `train` and early-stopped on `val`.
- **Target.** `exp_year / 20` during training; predictions are multiplied back by 20 and clipped to [0, 20].
- **Loss.** MSE by default; `--loss huber` (δ = 0.05 in scaled units, i.e. 1 year) for Question 3.
- **Long texts.** `--strategy head` by default. No CV has more than 162 DistilBERT tokens (stage 4), so with `max_len = 256` nothing is truncated and every job date reaches the model; the run logs `cvs_losing_a_year = 0`. `head_tail` and `chunk_avg` are implemented for longer CVs (checked with `max_len = 64`: head loses a year in 405 of 600 val CVs, chunk_avg in 0).
- **Optimisation.** AdamW, lr 5e-5, weight decay 0.01, linear schedule with 10% warmup, gradient clipping at 1.0. Batch 8 with gradient accumulation 2 (effective batch 16), padded per batch.
- **Stopping and hardware.** At most 8 epochs, early stopping on validation MAE with patience 2. MPS, fp32 (`PYTORCH_ENABLE_MPS_FALLBACK=1`), seed 42.

Training runs as a script in the Terminal, not in this notebook:

```bash
python -m src.train_bert                    # main model: clean OCR text
python -m src.train_bert --variant raw      # Question 3
```

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluation import collect_val_metrics
from src.visualization import plot_training_curve

def bert_metrics(run):
    for subdir in ("bert", "bert_tests"):
        path = utils.METRICS_DIR / subdir / f"{run}.json"
        if path.exists():
            return json.loads(path.read_text())
    return None

main_run = bert_metrics("bert__clean")
if main_run is None:
    print("Full training not done yet: showing the 200-CV / 1-epoch quick test.")
    main_run = bert_metrics("quick_test")
pd.Series({**main_run["config"], "effective_batch_size": main_run["effective_batch_size"],
           "epochs_trained": main_run["epochs_trained"], "best_epoch": main_run["best_epoch"],
           "train_minutes": round(main_run["train_seconds"] / 60, 1),
           "seconds_per_epoch": round(main_run["seconds_per_epoch"], 1),
           "inference_ms_per_cv": round(1000 * main_run["inference_seconds_per_cv"], 1),
           **main_run["hardware"]}, name="value").to_frame()

### Do the dates survive truncation?

Yes. The run reports, for train and validation, the longest CV in tokens, how many CVs are cut, and how many lose at least one year that the full text contains.

In [ ]:
pd.DataFrame(main_run["truncation"]).T

## Training curve

**Why this chart.** The training loss shows whether the model is still learning; the validation MAE shows whether that learning generalises, and early stopping keeps the epoch where it is lowest. They are drawn in two panels because they have different units. The dashed lines are the validation MAE of the TF-IDF baseline and of the date rules, for reference.

In [ ]:
history = pd.DataFrame(main_run["history"])
val = collect_val_metrics(("baselines",))
ref = val[val["variant"] == "clean"].set_index("model")["val_mae"]
fig = plot_training_curve(history, main_run["best_epoch"], {"TF-IDF": ref["tfidf"], "rules": ref["rules"]},
                          title=f"{main_run['run']} · {main_run['n_train']} training CVs")
plt.show()
history.round(4)

## Validation MAE: BERT vs the baselines

All models on the same validation CVs (the test set is still untouched). The table also covers the other text variants and losses once those runs exist.

In [ ]:
val = collect_val_metrics(("baselines", "bert"))
display(val.pivot(index="model", columns="variant", values="val_mae").round(3))
val[val["variant"] == "clean"].sort_values("val_mae").round(4)